# Sanity check fungsi evaluasi

Fungsi evaluasi dipakai semua model untuk menghitung Recall@10, NDCG@10, dan HitRate@10. Notebook ini mengecek rumusnya dengan contoh kecil yang bisa dihitung manual, lalu menjalankannya pada Popularity di seluruh evaluable users.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.evaluate import (hits_and_positions, recall_at_k, ndcg_at_k, hitrate_at_k,
                          sanity_check_metrics, evaluate_model, evaluate_per_user)
from src.protocol import build_protocol
from src.popularity import PopularityRecommender

D:\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Contoh hitungan manual

In [2]:
recs = ['x', 'a', 'y']
relevant_demo = {'a', 'z'}
print('posisi hit:', hits_and_positions(recs, relevant_demo, k=3))
print('Recall@3  :', recall_at_k(recs, relevant_demo, 3))
print('NDCG@3    :', round(ndcg_at_k(recs, relevant_demo, 3), 4))
print('HitRate@3 :', hitrate_at_k(recs, relevant_demo, 3))

posisi hit: [2]
Recall@3  : 0.5
NDCG@3    : 0.3869
HitRate@3 : 1.0


## Sanity check lima contoh

Lima contoh kecil (hit di posisi 2, hit di posisi 1, tidak ada hit, semua relevan di atas, dan hit di posisi 10) dicocokkan dengan hasil hitungan manual. Sel ini berhenti dengan error kalau ada satu saja yang tidak cocok.

In [3]:
sanity_check_metrics()

OK  1 hit di posisi 2 dari 2 relevan: Recall=0.5000 NDCG=0.3869 HitRate=1
OK  hit di posisi 1, 1 relevan: Recall=1.0000 NDCG=1.0000 HitRate=1
OK  tidak ada hit: Recall=0.0000 NDCG=0.0000 HitRate=0
OK  semua relevan di posisi teratas: Recall=1.0000 NDCG=1.0000 HitRate=1
OK  hit hanya di posisi 10: Recall=1.0000 NDCG=0.2891 HitRate=1
Semua contoh manual cocok.


True

## Muat protokol dan model Popularity

Evaluasi memakai protokol bersama (evaluable users dan kandidat warm dari `src/protocol.py`) dan model Popularity sebagai contoh.

In [4]:
train, val, cand_set, seen, relevant = build_protocol(str(ROOT / 'data') + '/')
pop = PopularityRecommender().fit(train, cand_set)
print('evaluable users:', len(relevant))

evaluable users: 11850


## Evaluasi pada semua evaluable users

`evaluate_model` meminta top-10 dari model untuk setiap evaluable user, lalu merata-ratakan Recall@10, NDCG@10, dan HitRate@10.

In [5]:
evaluate_model(pop, relevant, seen, k=10)

{'Recall@10': 0.01359501100582536,
 'NDCG@10': 0.009290692212411591,
 'HitRate@10': 0.031561181434599156,
 'n_users': 11850}

## Hasil per user

`evaluate_per_user` mengembalikan nilai tiap user, bukan hanya rata-ratanya. Data per user ini diperlukan untuk analisis per kelompok user (RQ2) dan untuk bootstrap confidence interval. Rata-ratanya harus sama dengan hasil di atas.

In [6]:
df = evaluate_per_user(pop, relevant, seen, k=10)
print(len(df), 'user')
display(df.head(3))
df[['recall', 'ndcg', 'hit']].mean()

11850 user


,user_id,n_relevant,recall,ndcg,hit
0,AE22B7BMXT7GAGY3XLHICMESNAPA,1,0.0,0.0,0.0
1,AE22HOJPKBRKRP6XGH6SU7EFDS6A,3,0.0,0.0,0.0
2,AE23JYHGEN3D35CHE5OQQYJOW5RA,5,0.0,0.0,0.0


recall    0.013595
ndcg      0.009291
hit       0.031561
dtype: float64